In [1]:
import os
import textwrap
import math
import json
import pandas as pd
import numpy as np
from collections import Counter
from datetime import datetime
from dateutil.relativedelta import relativedelta
from IPython.display import display
from IPython.display import Markdown

from langchain_openai import ChatOpenAI

import api


def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatOpenAI(
  openai_api_key = os.getenv('OPENAI_API_KEY'),
  model='gpt-4o-mini',
  temperature=1,
)

# 買進賣出需要的手續費
fee = {
  'tax_stock' : 0.003,          # 買進與賣出各0.003
  'tax_future' : 0.00002 * 2,   # 買進與賣出各0.00002
  'fee_stock' : 0.001425 * 2,   # 券商手續費公道價，買進與賣出各0.001425
  'fee_future' : 0.00002 * 2,   # 個家不大相同，以交易稅計算
  'sliding_price' : 0.00001,    # 滑價
}

env = {
    "stock_id" : "TSLA",
    "country" : "us",
    "start_date" : "20231001",
    "end_date" : "20240930",
    "path_out" : "out_stock/GA_sot_4omini/",
}

# 載入 factors

In [2]:
factors = {}
path_factors = f"{env['path_out']}{env['stock_id']}/factors.json"
print(path_factors)
with open(path_factors, "r") as f:
  factors = json.load(f)

# Access by index using list of keys
key_list = list(factors.keys())
value_list = list(factors.values())
print(key_list)
print(len(key_list))
print(len(value_list))

out_stock/GA_sot_4omini/TSLA/factors.json
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30']
30
30


# 延伸factors

In [3]:
def save_data(folder_path, data):
    count = 1
    while True:
        if os.path.exists(folder_path + str(count) + ".json"):
            count += 1
            continue
        else:
            out_file = folder_path + str(count) + ".json"
            break
    json_content = json.dumps(data, ensure_ascii=False, indent=4)
    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    with open(out_file, "w", encoding="UTF-8") as f:
        f.write(json_content)
    print(f"已儲存至{out_file}")

In [4]:
from api import factor_expanding


print("running... factor_expanding")
data = factor_expanding(llm, env, key_list, value_list)
# save_data(path_expand, data)

running... factor_expanding
Already processed for date 20240926, skip this date
Already processed for date 20240925, skip this date
Already processed for date 20240924, skip this date
Already processed for date 20240923, skip this date
Already processed for date 20240919, skip this date
Already processed for date 20240918, skip this date
Already processed for date 20240917, skip this date
Already processed for date 20240916, skip this date
Already processed for date 20240912, skip this date
Already processed for date 20240911, skip this date
Already processed for date 20240910, skip this date
Already processed for date 20240909, skip this date
Already processed for date 20240905, skip this date
Already processed for date 20240904, skip this date
Already processed for date 20240903, skip this date
Already processed for date 20240829, skip this date
Already processed for date 20240828, skip this date
Already processed for date 20240827, skip this date
Already processed for date 20240826,

In [ ]:
# 讀取舊檔
path_expand = f"{env['path_out']}{env['stock_id']}/expand1.json"
with open(path_expand, 'r') as f:
    old_data = json.load(f)
print(old_data["output_data"])

{'skeleton': {'1': {'response': 'Judgment: #no  \nReason: The news highlights various negative aspects surrounding Tesla, including a fire incident, missing technology in the Cybertruck, and an increase in headwinds ahead of the Robotaxi event, which suggests a bearish sentiment.', 'sig': -1, 'token': {'input_tokens': 599, 'output_tokens': 48, 'total_tokens': 647}}, '2': {'response': 'Judgment: #no  \nReason: The news contains several negative reports concerning Tesla, including fire incidents, potential underperformance of upcoming events, and rating downgrades, which could lead to investor concern.', 'sig': -1, 'token': {'input_tokens': 594, 'output_tokens': 41, 'total_tokens': 635}}, '3': {'response': 'Judgment: #no  \nReason: The news suggests a combination of negative events for Tesla, including a fire incident, a potential underwhelming Robotaxi event, and a rating downgrade, which could lead to investor concern.', 'sig': -1, 'token': {'input_tokens': 599, 'output_tokens': 45, 't

In [3]:
from api import eval

# 假設 individual, stock_id 和 data 是您要傳遞給 eval 函數的參數
individual = [1, 0, 1, 0, 1]
stock_id = 'AAPL'
data = {
    '20230101': {
        'skeleton': {
            '1': {'sig': 1},
            '2': {'sig': 0},
            '3': {'sig': -1},
            '4': {'sig': 1},
            '5': {'sig': 0}
        }
    }
}

result = eval(individual, stock_id, data)
print(result)

{'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0, 'avail_count': 0, 'ttl_count': 1, 'accuracy': 0, 'precision': 0, 'recall': 0, 'ev': 0, 'precision_ev': 0}
